In [ ]:
import pennylane as qml
import matplotlib.pyplot as plt
import numpy as np

def get_H(num_spins, k, h):
    """Construction function the ANNNI Hamiltonian (J=1)"""

    # Interaction between spins (neighbouring):
    H = -1 * (qml.PauliX(0) @ qml.PauliX(1))
    for i in range(1, num_spins - 1):
        H = H  - (qml.PauliZ(i) @ qml.PauliZ(i + 1))

    # Interaction between spins (next-neighbouring):
    for i in range(0, num_spins - 2):
        H = H + k * (qml.PauliZ(i) @ qml.PauliZ(i + 2))

    # Interaction of the spins with the magnetic field
    for i in range(0, num_spins):
        H = H - h * qml.PauliX(i)

    return H


In [ ]:
n_qubits = 8
hamiltonian = get_H(n_qubits, 0.5, 0.5)


In [ ]:
dev = qml.device("lightning.qubit", wires=n_qubits)


In [ ]:
def ansatz(params):
    qml.StronglyEntanglingLayers(weights=params, wires=range(n_qubits))

@qml.qnode(dev)
def circuit(params):
    ansatz(params)
    return qml.expval(hamiltonian)

@qml.qnode(dev)
def state_circuit(params):
    ansatz(params)
    return qml.state()

def cost(params):
    return circuit(params)


In [ ]:
opt = qml.AdamOptimizer(stepsize=0.1)
shape = qml.StronglyEntanglingLayers.shape(n_layers=2, n_wires=n_qubits)
rng = np.random.default_rng(12345)
params = rng.random(size=shape)

for i in range(100):
    params = opt.step(cost, params)

ground_state_vec = state_circuit(params)
real_state = np.real(ground_state_vec)
print(real_state)
